# SAM2 experiment in Google Colab

This notebook runs the repo's existing `scripts/tracking/track_players_sam2_experiment.py` CLI end to end in Colab.

## Optional: mount Google Drive

Turn this on if your repo or video lives in Drive, or if you want outputs to persist there.

In [ ]:
MOUNT_DRIVE = False

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

## Repo setup

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_NAME = 'computer-vision-pipeline'
REPO_DIR = Path('/content') / REPO_NAME
REPO_URL = ''  # Optional: set this if the repo is not already present in /content.
DRIVE_REPO_DIR = Path('/content/drive/MyDrive') / REPO_NAME

if REPO_DIR.exists():
    pass
elif DRIVE_REPO_DIR.exists():
    REPO_DIR = DRIVE_REPO_DIR
elif REPO_URL:
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    raise RuntimeError('Put the repo in /content, mount Drive with the repo in MyDrive, or set REPO_URL.')

os.chdir(REPO_DIR)
print(f'Working directory: {REPO_DIR}')

## Install dependencies

In [ ]:
import importlib.util
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-sam2.txt'])

## Roboflow key and video

Use a Colab secret named `ROBOFLOW_API_KEY` if you have one. Otherwise the cell will prompt without echoing the key.

Use upload for a quick local file, or point `VIDEO_PATH` to a file in `/content` or Google Drive.

In [ ]:
from getpass import getpass
from pathlib import Path
import os

try:
    from google.colab import files, userdata
except ImportError:
    files = None
    userdata = None

api_key = os.environ.get('ROBOFLOW_API_KEY', '').strip()
if not api_key and userdata is not None:
    try:
        api_key = userdata.get('ROBOFLOW_API_KEY').strip()
    except Exception:
        api_key = ''
if not api_key:
    api_key = getpass('Roboflow API key: ').strip()
if not api_key:
    raise RuntimeError('ROBOFLOW_API_KEY is required.')
os.environ['ROBOFLOW_API_KEY'] = api_key

UPLOAD_VIDEO = True
VIDEO_PATH = ''  # Example: /content/game.mp4 or /content/drive/MyDrive/game.mp4

if UPLOAD_VIDEO:
    if files is None:
        raise RuntimeError('google.colab.files is unavailable. Set UPLOAD_VIDEO = False and point VIDEO_PATH to a local file.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No video uploaded.')
    VIDEO_PATH = str(Path('/content') / next(iter(uploaded)))

video_path = Path(VIDEO_PATH)
if not video_path.exists():
    raise RuntimeError(f'Video not found: {video_path}')
print(f'Video: {video_path}')

## Run the SAM2 experiment

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

FRAME_STRIDE = 15
MAX_FRAMES = 300
CONFIDENCE = 0.0
SAM2_MODEL_ID = 'facebook/sam2-hiera-large'
SAM2_MATCH_IOU = 0.15

output_video = Path('outputs') / f'{video_path.stem}_sam2.mp4'
tracks_jsonl = Path('outputs') / f'{video_path.stem}_sam2_tracks.jsonl'
inference_jsonl = Path('outputs') / f'{video_path.stem}_sam2_inference.jsonl'

command = [
    sys.executable,
    'scripts/tracking/track_players_sam2_experiment.py',
    str(video_path),
    '--frame-stride', str(FRAME_STRIDE),
    '--max-frames', str(MAX_FRAMES),
    '--confidence', str(CONFIDENCE),
    '--sam2-model-id', SAM2_MODEL_ID,
    '--sam2-match-iou', str(SAM2_MATCH_IOU),
    '--output', str(output_video),
    '--tracks-output', str(tracks_jsonl),
    '--json-output', str(inference_jsonl),
]
print(' '.join(shlex.quote(part) for part in command))
subprocess.check_call(command)
print(f'Tracked video: {output_video}')
print(f'Tracks JSONL: {tracks_jsonl}')
print(f'Inference JSONL: {inference_jsonl}')

## Optional summary

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, 'scripts/tracking/summarize_tracking_run.py', str(tracks_jsonl)])

## Preview and download outputs

In [ ]:
from IPython.display import Video, display

display(Video(str(output_video), embed=True))

In [ ]:
SAVE_OUTPUTS_TO_DRIVE = False
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/sam2_outputs')
DOWNLOAD_OUTPUTS = True

if SAVE_OUTPUTS_TO_DRIVE:
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for path in (output_video, tracks_jsonl, inference_jsonl):
        target = DRIVE_OUTPUT_DIR / path.name
        target.write_bytes(path.read_bytes())
        print(f'Saved to Drive: {target}')

if DOWNLOAD_OUTPUTS:
    if files is None:
        raise RuntimeError('google.colab.files is unavailable. Download manually from the file browser.')
    for path in (output_video, tracks_jsonl, inference_jsonl):
        files.download(str(path))